In [ ]:
import random
from collections import deque
from itertools import combinations


class TrafficLight:
    def __init__(self, id):
        self.id = id
        self.state = 'red'
        self.green_time = 30
        self.red_time = 30


class TrafficSignalScheduler:
    def __init__(self):
        self.traffic_lights = {
            f'light{i}': TrafficLight(f'light{i}')
            for i in range(1, 5)
        }

        self.emergency_queue = deque()

        self.lane_density = {
            'light1': 'low',
            'light2': 'medium',
            'light3': 'high',
            'light4': 'traffic jam'
        }

    def add_emergency_vehicle(self, vehicle_id):
        self.emergency_queue.append(vehicle_id)

    def calculate_total_loss(self):
        penalty_map = {
            "traffic jam": 20,
            "high": 15,
            "medium": 10,
            "low": 5
        }

        total_loss = 0

        for light in self.traffic_lights.values():
            density = self.lane_density[light.id]
            penalty = penalty_map[density]

            if light.state == "red":
                total_loss += penalty
            else:
                total_loss += penalty * 0.3

        return total_loss

    def generate_feasible_array(self):
        return [
            light_id for light_id, light in self.traffic_lights.items()
            if light.state == 'red'
        ]

    def select_best_lights(self, feasible_array):
        best_lights = []
        min_loss = float('inf')

        for i in range(1, len(feasible_array) + 1):
            for combination in combinations(feasible_array, i):
                original_states = {
                    lid: self.traffic_lights[lid].state
                    for lid in combination
                }

                for lid in combination:
                    self.traffic_lights[lid].state = 'green'

                potential_loss = self.calculate_total_loss()

                for lid in combination:
                    self.traffic_lights[lid].state = original_states[lid]

                if potential_loss < min_loss:
                    min_loss = potential_loss
                    best_lights = combination

        return best_lights

    def handle_emergency(self):
        if not self.emergency_queue:
            return

        priority_order = ["traffic jam", "high", "medium", "low"]

        best_light = max(
            self.traffic_lights.keys(),
            key=lambda lid: priority_order.index(self.lane_density[lid])
        )

        light = self.traffic_lights[best_light]
        light.state = "green"

        print(f"Emergency priority given to {best_light}")
        self.emergency_queue.popleft()

    def handle_normal_traffic(self, light_id):
        light = self.traffic_lights[light_id]
        density = self.lane_density[light_id]

        if density == "traffic jam":
            light.state = "green"
            light.green_time += 15

        elif density == "high":
            light.state = "green"
            light.green_time += 10

        elif density == "medium":
            light.state = random.choice(["green", "red"])

        else:
            light.state = "red"

    def get_current_signals(self):
        return {
            light.id: {
                "color": light.state,
                "time": light.green_time if light.state == "green" else light.red_time
            }
            for light in self.traffic_lights.values()
        }

    def schedule_traffic_lights(self, cycles=1):
        for _ in range(cycles):

            if self.emergency_queue:
                self.handle_emergency()
            else:
                feasible = self.generate_feasible_array()
                best_lights = self.select_best_lights(feasible)

                for lid in best_lights:
                    self.handle_normal_traffic(lid)

            print("Current Signals:", self.get_current_signals())
            print("Total Loss:", self.calculate_total_loss())
            print("-" * 50)


if __name__ == "__main__":
    scheduler = TrafficSignalScheduler()
    scheduler.add_emergency_vehicle("Ambulance1")
    scheduler.schedule_traffic_lights(cycles=3)